In [13]:
#Importing and reading crime data

import pandas as pd
df = pd.read_csv("crime_data/Major_Crime_Indicators_Data.csv")
print(df.columns.tolist())
print(len(df))

['OBJECTID', 'EVENT_UNIQUE_ID', 'REPORT_DATE', 'OCC_DATE', 'REPORT_YEAR', 'REPORT_MONTH', 'REPORT_DAY', 'REPORT_DOY', 'REPORT_DOW', 'REPORT_HOUR', 'OCC_YEAR', 'OCC_MONTH', 'OCC_DAY', 'OCC_DOY', 'OCC_DOW', 'OCC_HOUR', 'DIVISION', 'LOCATION_TYPE', 'PREMISES_TYPE', 'UCR_CODE', 'UCR_EXT', 'OFFENCE', 'CSI_CATEGORY', 'HOOD_158', 'NEIGHBOURHOOD_158', 'HOOD_140', 'NEIGHBOURHOOD_140', 'LONG_WGS84', 'LAT_WGS84', 'x', 'y']
474819


In [14]:
# ML model trained using supervised learning
# Predicts next month's crime count per Toronto neighbourhood

import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
import json

# ---------- 1. Load & clean ----------
df = pd.read_csv("crime_data/Major_Crime_Indicators_Data.csv")

df = df.dropna(subset=["OCC_YEAR", "OCC_MONTH", "NEIGHBOURHOOD_158"])
df["OCC_MONTH_NUM"] = pd.to_datetime(df["OCC_MONTH"], format="%B").dt.month
df["OCC_YEAR"] = df["OCC_YEAR"].astype(int)

df = df[df["OCC_YEAR"] >= 2014]             # trim occurrences before 2014
df = df[df["NEIGHBOURHOOD_158"] != "NSA"]   # drop unassigned pseudo-hood

df["date"] = pd.to_datetime(
    dict(year=df["OCC_YEAR"], month=df["OCC_MONTH_NUM"], day=1)
)

# Crime Count per neighbourhood per month
panel = (df.groupby(["NEIGHBOURHOOD_158", "date"]).size()
           .rename("count").reset_index())

# fill missing months with 0 so lags are honest
panel = (panel.set_index("date")
              .groupby("NEIGHBOURHOOD_158")["count"]
              .resample("MS").sum().fillna(0).reset_index())

# drop partial months
panel = panel[panel["date"] < panel["date"].max()]
panel = panel.sort_values(["NEIGHBOURHOOD_158", "date"]).reset_index(drop=True)

# Lag features
g = panel.groupby("NEIGHBOURHOOD_158")["count"]
panel["lag_1"]  = g.shift(1)
panel["lag_3m"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
panel["lag_12"] = g.shift(12)
panel["month"]  = panel["date"].dt.month
panel["target"] = g.shift(-1)          # next month = what we predict
panel = panel.dropna()

# Temporal holdout split + train
cutoff = panel["date"].max() - pd.DateOffset(months=6)
train = panel[panel["date"] <= cutoff]
test  = panel[panel["date"] >  cutoff]
feats = ["count", "lag_1", "lag_3m", "lag_12", "month"]

model = HistGradientBoostingRegressor(max_iter=300, random_state=42)
model.fit(train[feats], train["target"])

mae = mean_absolute_error(test["target"], model.predict(test[feats]))
print(f"Rows trained on: {len(train):,}")
print(f"Test MAE (6-month temporal holdout): {mae:.2f} incidents/neighbourhood/month")

# Predict next month for every neighbourhood, export
latest = panel[panel["date"] == panel["date"].max()].copy()
latest["predicted"] = model.predict(latest[feats])
latest["risk"] = (
    (latest["predicted"] - latest["predicted"].min())
    / (latest["predicted"].max() - latest["predicted"].min())
).round(4) # normalized 0–1 risk score

out = (latest[["NEIGHBOURHOOD_158", "predicted", "risk"]]
       .rename(columns={"NEIGHBOURHOOD_158": "name"})
       .sort_values("risk", ascending=False)
       .round({"predicted": 1})
       .to_dict(orient="records"))

json.dump(out, open("predictions.json", "w"), indent=2)

print(f"\nExported predictions.json ({len(out)} neighbourhoods)")
print("\nTop 10 predicted risk:")
for r in out[:10]:
    print(f"  {r['name']}: {r['predicted']} predicted incidents (risk {r['risk']})")


# Export recent incident points for the dots layer
recent = df[df["date"] >= df["date"].max() - pd.DateOffset(months=3)].copy()

# filter privacy-offset junk / zero coordinates
recent = recent[(recent["LAT_WGS84"].between(43.4, 44.0)) &
                (recent["LONG_WGS84"].between(-79.8, -79.0))]

# cap for map performance; sample keeps the spatial distribution
if len(recent) > 5000:
    recent = recent.sample(5000, random_state=42)

points = recent[["LAT_WGS84", "LONG_WGS84", "OFFENCE", "OCC_DATE"]] \
    .rename(columns={"LAT_WGS84": "lat", "LONG_WGS84": "lng",
                     "OFFENCE": "offence", "OCC_DATE": "date"}) \
    .to_dict(orient="records")

json.dump(points, open("crime_points.json", "w"))
print(f"Exported {len(points)} incident points (last 3 months)")

Rows trained on: 20,066
Test MAE (6-month temporal holdout): 5.13 incidents/neighbourhood/month

Exported predictions.json (158 neighbourhoods)

Top 10 predicted risk:
  Mimico-Queensway (160): 83.3 predicted incidents (risk 1.0)
  West Humber-Clairville (1): 77.8 predicted incidents (risk 0.9287)
  Moss Park (73): 69.0 predicted incidents (risk 0.8172)
  Annex (95): 68.0 predicted incidents (risk 0.804)
  Downtown Yonge East (168): 61.1 predicted incidents (risk 0.7159)
  Yonge-Bay Corridor (170): 60.9 predicted incidents (risk 0.7126)
  York University Heights (27): 56.4 predicted incidents (risk 0.6553)
  Kensington-Chinatown (78): 45.1 predicted incidents (risk 0.5105)
  Clairlea-Birchmount (120): 43.7 predicted incidents (risk 0.492)
  Yorkdale-Glen Park (31): 42.5 predicted incidents (risk 0.4771)
Exported 5000 incident points (last 3 months)


In [15]:
import json, re

geo = json.load(open("crime_data/Neighbourhoods.geojson"))
preds = json.load(open("predictions.json"))

# Extract the numeric ID from "Mimico-Queensway (160)" -> "160"
def hood_id(name):
    m = re.search(r"\((\d+)\)", name)
    return m.group(1) if m else None

top10 = {hood_id(p["name"]): p for p in preds[:10]}

# City file: check props with -> print(geo["features"][0]["properties"].keys())
# The ID field is usually AREA_LONG_CODE or AREA_SHORT_CODE ("160" or "0160")
features = []
for f in geo["features"]:
    props = f["properties"]
    code = str(props.get("AREA_SHORT_CODE") or props.get("AREA_LONG_CODE", "")).lstrip("0")
    if code in top10:
        features.append({
            "type": "Feature",
            "properties": {
                "name": top10[code]["name"].split(" (")[0],
                "risk": top10[code]["risk"],
            },
            "geometry": f["geometry"],
        })

json.dump({"type": "FeatureCollection", "features": features},
          open("high_risk_zones.json", "w"))
print(f"Wrote {len(features)} zones")   # 10 zones

Wrote 10 zones
